# 1. Import Modules

In [1]:
import csv
import json
import requests
import yaml
from datetime import datetime

# 2. Set Up & Authentication

In [2]:
#Get the current date. This will be used to name the files created.
report_date = datetime.now().strftime("%m_%d_%Y")

#Safe loads the secrets file containing your username and password.
with open("../secrets.yml") as f:
    secrets = yaml.safe_load(f)

#Credentials to post for authentication
baseURL = 'https://archives.pratt.edu/staff/api'
user = secrets['username']
password = secrets['password']
repository = "2" #The Pratt Archives has only one repository which will always be "2"

#Sends the authentication request to the API
auth = requests.post(baseURL + '/users/' + user + '/login?password='+ password).json()

#If authentication fails, an error will be printed.
if 'session' not in auth:
    print("Error: authentication failed.")
    print(auth)
else:
    session = auth['session']
    headers = {'X-ArchivesSpace-Session': session,
            'Content_Type': 'application/json'}
    
    print('Authentication successful.')

Authentication successful.


# 3. Get Accessions and Dump Into JSON
Note: It is normal for this code block to take a while (~9 minutes) because of the large number of accessions. 

In [3]:
#Endpoint for Get a list of Accessions for a Repository
endpoint = '/repositories/' + repository + '/accessions?all_ids=true'

#Get IDs for the accessions
ids = requests.get(baseURL + endpoint, headers=headers).json()
print(f"{len(ids)} accessions found.")

#Iterate over IDs
records = []

print("Retrieving accession information...")
counter = 1
for id in ids:
    endpoint = '/repositories/' + repository + '/accessions/' + str(id)
    output = requests.get(baseURL + endpoint, headers=headers).json()
    records.append(output)

    if counter % 50 == 0: 
        print(f"Status: {counter}/{len(ids)}")

    counter +=1 

#Data is dumped into a JSON file
with open(f'../output/accessions_{report_date}.json', 'w') as f:
    json.dump(records, f)

print(f"Completed.")

632 accessions found.
Retrieving accession information...
Status: 50/632
Status: 100/632
Status: 150/632
Status: 200/632
Status: 250/632
Status: 300/632
Status: 350/632
Status: 400/632
Status: 450/632
Status: 500/632
Status: 550/632
Status: 600/632
Completed.


# 4. Create Column Headers for CSV file

In [5]:
with open(f'../output/accessions_{report_date}.json', 'r') as json_file:
    json_data = json.load(json_file)

csv_fields = []

#Top-level keys from the JSON file
base_fieldnames = [
    "uri",
    "title",
    "publish",
    "content_description",
    "condition_description",
    "disposition",
    "inventory",
    "provenance",
    "accession_date",
    "restrictions_apply",
    "access_restrictions",
    "access_restrictions_note",
    "use_restrictions",
    "id_0",
    "id_1",
    "acquisition_type",
    "resource_type"
]
csv_fields.extend(base_fieldnames)

#Add new field for concatenating Accession ID
csv_fields.extend(["identifier"])

#Fields that are nested within the "collection_management" dictionary.
nested_fieldnames = [
    "processing_priority",
    "processing_status",
    "processors",
]
csv_fields.extend(nested_fieldnames)

#Create columns based on the maximum number of extents and dates that appear for each accession record
max_extents = max(len(r.get("extents", [])) for r in json_data)
max_dates = max(len(r.get("dates", [])) for r in json_data)

for i in range(1, max_extents + 1):
    csv_fields.extend([
        f"extent_{i}_container_summary",
        f"extent_{i}_number",
        f"extent_{i}_portion",
        f"extent_{i}_extent_type"
    ])

for i in range(1, max_dates + 1):
    csv_fields.extend([
        f"date_{i}_expression",
        f"date_{i}_begin",
        f"date_{i}_end",
        f"date_{i}_date_type",
        f"date_{i}_label"
    ])

print(f"Columns to be added to CSV: {json.dumps(csv_fields, indent=2)}")

Columns to be added to CSV: [
  "uri",
  "title",
  "publish",
  "content_description",
  "condition_description",
  "disposition",
  "inventory",
  "provenance",
  "accession_date",
  "restrictions_apply",
  "access_restrictions",
  "access_restrictions_note",
  "use_restrictions",
  "id_0",
  "id_1",
  "acquisition_type",
  "resource_type",
  "identifier",
  "processing_priority",
  "processing_status",
  "processors",
  "extent_1_container_summary",
  "extent_1_number",
  "extent_1_portion",
  "extent_1_extent_type",
  "extent_2_container_summary",
  "extent_2_number",
  "extent_2_portion",
  "extent_2_extent_type",
  "extent_3_container_summary",
  "extent_3_number",
  "extent_3_portion",
  "extent_3_extent_type",
  "extent_4_container_summary",
  "extent_4_number",
  "extent_4_portion",
  "extent_4_extent_type",
  "extent_5_container_summary",
  "extent_5_number",
  "extent_5_portion",
  "extent_5_extent_type",
  "extent_6_container_summary",
  "extent_6_number",
  "extent_6_porti

# 5. Write JSON to CSV
This writes a basic CSV file using the fieldnames listed above. For Accession records with multiple dates and extents, this data will be iterated out over sequential columns (example: date_1_begin, date_2_begin, date_3_begin, etc.)

In [6]:
with open(f'../output/accessions_{report_date}.json', 'r') as json_file:
    json_data = json.load(json_file)

with open(f'../output/accessions_{report_date}.csv', 'w', newline="") as csv_file:
    csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
    csv_writer.writeheader()

    for entry in json_data:

        row = {}

        #Retrieves top-level fields.
        for b in base_fieldnames:
            row[b] = entry.get(b, "")

        #Concatenates IDs into one field.
        if row['id_1'] != "": 
            row['identifier'] = f"{row['id_0']}-{row['id_1']}"
        else:
            row['identifier'] = row['id_0']

        #Retrieves data for fields that are nested within the "collection_management" dictionary.
        for n in nested_fieldnames:
            if entry.get('collection_management') is not None:
                row[n] = entry.get('collection_management').get(n)
            else: 
                row[n] = None

        # Retrieves extents.
        for i, extent in enumerate(entry.get("extents", []), start=1):
            row[f"extent_{i}_container_summary"] = extent.get("container_summary")
            row[f"extent_{i}_number"] = extent.get("number")
            row[f"extent_{i}_portion"] = extent.get("portion")
            row[f"extent_{i}_extent_type"] = extent.get("extent_type")

        # Retrieves dates.
        for i, date in enumerate(entry.get("dates", []), start=1):
            row[f"date_{i}_expression"] = date.get("expression")
            row[f"date_{i}_begin"] = date.get("begin")
            row[f"date_{i}_end"] = date.get("end")
            row[f"date_{i}_date_type"] = date.get("date_type")
            row[f"date_{i}_label"] = date.get("label")
        
        csv_writer.writerow(row)

